In [31]:
# Import the libraries 
import pandas as pd
import numpy as np

In [32]:
# Load the dataset
df = pd.read_csv('../data/bank_raw.csv', sep = ';')

# First look
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [33]:
# Overview data structure 
print('Shape: ', df.shape)
df.info()

Shape:  (45211, 17)
<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 8.1 MB


No null values, though we are going to check for sure.

In [34]:
# Null values
df.isnull().sum().sum()

np.int64(0)

In [35]:
# Replace 'unknown' values with null, we know this from data information on Kaggle
cols_with_unknown = ['job', 'education', 'contact', 'poutcome']

for col in cols_with_unknown:
    df[col] = df[col].replace('unknown', np.nan)

# Check for null values again
df.isnull().sum()

age              0
job            288
marital          0
education     1857
default          0
balance          0
housing          0
loan             0
contact      13020
day              0
month            0
duration         0
campaign         0
pdays            0
previous         0
poutcome     36959
y                0
dtype: int64

In [36]:
# Describe the numerical values
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [37]:
# Day and Month unique values
print(df['day'].unique())
print(df['month'].unique())

[ 5  6  7  8  9 12 13 14 15 16 19 20 21 23 26 27 28 29 30  2  3  4 11 17
 18 24 25  1 10 22 31]
<ArrowStringArray>
['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'jan', 'feb', 'mar', 'apr',
 'sep']
Length: 12, dtype: str


In [38]:
# Duration == 0?
(df['duration'] == 0).sum()

np.int64(3)

In [39]:
# Remove the 3 rows where duration is 0
df = df[df['duration'] > 0].copy()

# Check again
(df['duration'] == 0).sum()

np.int64(0)

In [40]:
# Check for consistency 
inconsistent = df[(df['pdays'] == -1) & (df['previous'] > 0)]
inconsistent.shape[0]

0

In [41]:
# Inspect target variable 
df['y'].value_counts()

y
no     39919
yes     5289
Name: count, dtype: int64

In [42]:
# Check for duplicates 
df.duplicated().sum()

np.int64(0)

In [43]:
# Create a Flag for previously contacted customers
df['prev_contacted'] = (df['pdays'] != -1).astype(int)

df['prev_contacted'].value_counts()

prev_contacted
0    36951
1     8257
Name: count, dtype: int64

0 = have not been contacted

1 = have been contacted

In [44]:
# Missing values percentage
df.isnull().mean() * 100

age                0.000000
job                0.637055
marital            0.000000
education          4.107680
default            0.000000
balance            0.000000
housing            0.000000
loan               0.000000
contact           28.798000
day                0.000000
month              0.000000
duration           0.000000
campaign           0.000000
pdays              0.000000
previous           0.000000
poutcome          81.746594
y                  0.000000
prev_contacted     0.000000
dtype: float64

In [45]:
# Drop rows with missing values in job and education
df = df.dropna(subset = ['education', 'job'])

df.isna().mean() * 100

age                0.000000
job                0.000000
marital            0.000000
education          0.000000
default            0.000000
balance            0.000000
housing            0.000000
loan               0.000000
contact           28.444084
day                0.000000
month              0.000000
duration           0.000000
campaign           0.000000
pdays              0.000000
previous           0.000000
poutcome          81.692521
y                  0.000000
prev_contacted     0.000000
dtype: float64

In [46]:
# Fill poutcome and contact with 'missing' label again
df['contact'] = df['contact'].fillna('missing')
df['poutcome'] = df['poutcome'].fillna('missing')

# Check all null values again
df.isnull().sum().sum()

np.int64(0)

In [47]:
# Create cat columns
cat_cols = df.select_dtypes(include = ['object', 'category']).columns

for col in cat_cols:
    print(f'Column: {col}')
    print(f'Unique values ({df[col].nunique()}): {df[col].unique()}')
    print('-' * 30)

Column: job
Unique values (11): <ArrowStringArray>
[   'management',    'technician',  'entrepreneur',       'retired',
        'admin.',      'services',   'blue-collar', 'self-employed',
    'unemployed',     'housemaid',       'student']
Length: 11, dtype: str
------------------------------
Column: marital
Unique values (3): <ArrowStringArray>
['married', 'single', 'divorced']
Length: 3, dtype: str
------------------------------
Column: education
Unique values (3): <ArrowStringArray>
['tertiary', 'secondary', 'primary']
Length: 3, dtype: str
------------------------------
Column: default
Unique values (2): <ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str
------------------------------
Column: housing
Unique values (2): <ArrowStringArray>
['yes', 'no']
Length: 2, dtype: str
------------------------------
Column: loan
Unique values (2): <ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str
------------------------------
Column: contact
Unique values (3): <ArrowStringArray>
['mi

/tmp/ipykernel_58354/285031923.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include = ['object', 'category']).columns


In [48]:
# Check the first few rows
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,prev_contacted
0,58,management,married,tertiary,no,2143,yes,no,missing,5,may,261,1,-1,0,missing,no,0
1,44,technician,single,secondary,no,29,yes,no,missing,5,may,151,1,-1,0,missing,no,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,missing,5,may,76,1,-1,0,missing,no,0
5,35,management,married,tertiary,no,231,yes,no,missing,5,may,139,1,-1,0,missing,no,0
6,28,management,single,tertiary,no,447,yes,yes,missing,5,may,217,1,-1,0,missing,no,0


In [49]:
# Save the new dataset
df.to_csv('../data/bank_cleaned.csv', index = False)